<a href="https://colab.research.google.com/github/Dr-Isam-ALJAWARNEH/fds-project-airnav/blob/main/Merge_All_NYC_Files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Below is a complete Google Colab code snippet that lets you upload your five CSV files and one GeoJSON file, merges the CSV files, reads the GeoJSON, performs a merge (using a common key if available or simply attaches a default geometry), and then converts the result into a Table abstraction using the datascience library.

Copy and paste the code below into a Colab cell and run it. When prompted, upload the following files exactly as named:

NYC_AQ.csv

NYC_PM.csv

NYC_PM_Part1.csv

NYC_PM_Part2.csv

NYC_PM_Part3.csv

nyc_polygon.geojson

What This Code Does:
File Upload:
Prompts you to upload all required files. It prints a list so you know what files to upload.

CSV Processing:
Reads each CSV file. If a file is missing (like NYC_AQ.csv), it prints a warning and continues with the rest.

GeoJSON Handling:
Reads the GeoJSON file from a temporary file, then removes that temporary file.

Merging Strategy:
Attempts to merge on a common key (if columns like polygon_id in the CSV and id in the GeoJSON exist). Otherwise, it attaches the first geometry from the GeoJSON to each CSV row.

Table Conversion:
Converts the final merged DataFrame into a datascience Table abstraction and displays it.

If you expect NYC_AQ.csv to be part of your analysis, make sure to include it when uploading your files. Otherwise, the code will simply process the ones you uploaded.

In [17]:
import io
import tempfile
import os
import pandas as pd
import geopandas as gpd
from datascience import Table
from google.colab import files

In [18]:
# Prompt the user to upload the files
print("Please upload the CSV and GeoJSON files as prompted.")
uploaded = files.upload()

Please upload the CSV and GeoJSON files as prompted.


Saving NYC_AQ.csv to NYC_AQ (2).csv
Saving NYC_PM.csv to NYC_PM (1).csv
Saving NYC_PM_Part1.csv to NYC_PM_Part1 (1).csv
Saving NYC_PM_Part2.csv to NYC_PM_Part2 (1).csv
Saving NYC_PM_Part3.csv to NYC_PM_Part3 (1).csv
Saving nyc_polygon.geojson to nyc_polygon (1).geojson


In [20]:
# List of expected CSV file names
expected_csv_files = [
    'NYC_AQ (2).csv',
    'NYC_PM (1).csv',
    'NYC_PM_Part1 (1).csv',
    'NYC_PM_Part2 (1).csv',
    'NYC_PM_Part3 (1).csv'
]

In [21]:
# Read and concatenate CSV files
df_list = []
for file in expected_csv_files:
    if file in uploaded:
        print(f"Reading {file} ...")
        df = pd.read_csv(io.BytesIO(uploaded[file]))
        df_list.append(df)
    else:
        print(f"Warning: {file} not uploaded.")

if df_list:
    df_csv = pd.concat(df_list, ignore_index=True)
else:
    raise ValueError("No CSV files were loaded.")

Reading NYC_AQ (2).csv ...
Reading NYC_PM (1).csv ...
Reading NYC_PM_Part1 (1).csv ...
Reading NYC_PM_Part2 (1).csv ...
Reading NYC_PM_Part3 (1).csv ...


In [23]:
# Read the GeoJSON file
geojson_filename = 'nyc_polygon (1).geojson'
if geojson_filename in uploaded:
    print(f"Reading {geojson_filename} ...")
    # Write the GeoJSON bytes to a temporary file so that GeoPandas can read it
    with tempfile.NamedTemporaryFile(suffix='.geojson', delete=False) as tmp:
        tmp.write(uploaded[geojson_filename])
        geojson_path = tmp.name
    gdf = gpd.read_file(geojson_path)
    os.remove(geojson_path)  # Clean up the temporary file
else:
    print(f"Warning: {geojson_filename} not uploaded.")
    gdf = None

Reading nyc_polygon (1).geojson ...


In [24]:
# Merge CSV data with GeoJSON based on a common key if available,
# otherwise attach a default geometry from the GeoJSON to each row.
if gdf is not None:
    if 'polygon_id' in df_csv.columns and 'id' in gdf.columns:
        print("Merging using common key 'polygon_id' and 'id'...")
        df_merged = pd.merge(df_csv, gdf, left_on='polygon_id', right_on='id', how='left')
    else:
        print("No common key found. Attaching a default geometry from the GeoJSON...")
        if not gdf.empty:
            df_csv['geometry'] = gdf.geometry.iloc[0]
        else:
            df_csv['geometry'] = None
        df_merged = df_csv
else:
    print("GeoJSON file was not uploaded; proceeding with CSV data only.")
    df_merged = df_csv

No common key found. Attaching a default geometry from the GeoJSON...


In [27]:
# Convert the merged DataFrame to a Table abstraction from the datascience library
table = Table.from_df(df_merged)
print("Displaying the Table abstraction:")
table.show(3)

Displaying the Table abstraction:


SensorID,time,latitude,longitude,bin0,bin1,bin2,bin3,bin4,bin5,bin6,bin7,bin8,bin9,bin10,bin11,bin12,bin13,bin14,bin15,bin16,bin17,bin18,bin19,bin20,bin21,bin22,bin23,temperature,humidity,pm25,pm1,pm10,geometry
NYCP2_CS01A,1631277304,40.8477,-73.8693,11,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,23.7,57.3,4.50881,nan,nan,"POLYGON ((-73.84859700000018 40.871670000000115, -73.845 ..."
NYCP2_CS01A,1631277308,40.8477,-73.8693,22,4,1,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,23.7,57.8,5.46242,nan,nan,"POLYGON ((-73.84859700000018 40.871670000000115, -73.845 ..."
NYCP2_CS01A,1631277313,40.8476,-73.8694,40,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,23.7,57.8,5.15488,nan,nan,"POLYGON ((-73.84859700000018 40.871670000000115, -73.845 ..."
